In [ ]:

from langchain_huggingface.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate


from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

In [3]:
from datasets import load_dataset

# Load the complete dataset
dataset = load_dataset("ankithreddy/repairdataset-mini")

# Access examples
example = dataset['train'][0]
image = example['image_path']  # PIL Image object
instruction = example['text']  # Repair instruction text
device = example['device_name']  # Target device

# Filter by category
repair_guides = dataset['train'].filter(lambda x: x['type'] == 'guide_overview')
teardowns = dataset['train'].filter(lambda x: x['type'] == 'teardown_analysis')
steps = dataset['train'].filter(lambda x: x['type'] == 'step_instruction')

In [ ]:
# Load the Qwen3-8B model - the larger, more powerful model
print("📥 Loading Qwen2.5-8B model...")
print("⏳ This is a larger model and will take longer to load...")

qwen_model_name = "Qwen/Qwen2.5-7B-Instruct"  # Using 7B variant

# Load tokenizer and model
qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_model_name)
qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype="auto",
    device_map="auto"
)

# Create pipeline
qwen_pipe = pipeline(
    "text-generation",
    model=qwen_model,
    tokenizer=qwen_tokenizer,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
)

# Wrap for LangChain
qwen_llm = HuggingFacePipeline(pipeline=qwen_pipe)

print("✅ Qwen3 model loaded successfully!")
print(f"📊 Model size: ~7-8 billion parameters")